# Model fitting for learning analysis

In [1]:
# Install packages
import Pkg
Pkg.add(url="https://github.com/ndawlab/em.git/")
Pkg.add("GLM")
Pkg.add("DataFrames")
Pkg.add("StatsBase")
Pkg.add("SpecialFunctions")
Pkg.add("JLD2")
Pkg.add("LogExpFunctions")
Pkg.add("DataFramesMeta")

    Updating git-repo `https://github.com/ndawlab/em.git`
    Updating registry at `~/.julia/registries/General.toml`
   Resolving package versions...
  No Changes to `~/.julia/environments/v1.11/Project.toml`
  No Changes to `~/.julia/environments/v1.11/Manifest.toml`
   Resolving package versions...
  No Changes to `~/.julia/environments/v1.11/Project.toml`
  No Changes to `~/.julia/environments/v1.11/Manifest.toml`
   Resolving package versions...
  No Changes to `~/.julia/environments/v1.11/Project.toml`
  No Changes to `~/.julia/environments/v1.11/Manifest.toml`
   Resolving package versions...
  No Changes to `~/.julia/environments/v1.11/Project.toml`
  No Changes to `~/.julia/environments/v1.11/Manifest.toml`
   Resolving package versions...
  No Changes to `~/.julia/environments/v1.11/Project.toml`
  No Changes to `~/.julia/environments/v1.11/Manifest.toml`
   Resolving package versions...
  No Changes to `~/.julia/environments/v1.11/Project.toml`
  No Changes to `~/.julia/envi

In [2]:
# Do some setup

full = false    # maintain full covariance matrix (vs a diagional one) at the group level
emtol = 1e-3    # stopping condition (relative change) for EM

# Load EM package
using EM

# Load other packages
using Statistics
using Random
using GLM
using DataFrames
using CSV
using StatsBase
using SpecialFunctions
using JLD2
using LogExpFunctions
using DataFramesMeta

# Create folder for saving results
to_save_folder = "model_fitting_results"
if !isdir(to_save_folder)
    mkdir(to_save_folder)
end

include("model_fitting_funs.jl")

fit_model_em (generic function with 1 method)

In [11]:
# Load the data for fitting
data = CSV.read("modeling_QG.csv", DataFrame)

# Sort by player and drop subjects with less than two games
data_sorted = sort(data, :black)
data_filtered = filter(sdf -> nrow(sdf) ≥ 2, groupby(data_sorted, :black))
data_filtered = vcat(data_filtered...)

# Count total number of players
total_num_subjects = length(unique(data_filtered.black))

# Subsample subjects from the data and rename
data_subset = data_filtered[data_filtered.black .< 20001, :]
rename!(data_subset, [:sub, :c, :r])

Row,sub,c,r
,Int64,Int64,Int64
1,1,2,1
2,1,2,1
3,2,2,1
4,2,2,-1
5,2,2,-1
6,2,2,1
7,2,2,1
8,2,2,-1
9,3,2,1


In [12]:
# Count the number of players in the subset
num_subjects = length(unique(data_subset.sub))

12481

In [37]:
# Modified likelihood function
function qlik(params, data; include_rl = true, include_wsls = true)

    # Bias: beta
    beta_1 = params[1]
    p_idx = 2;

    # Rescorla-Wagner: beta and learning rate
    if (include_rl)
        beta_rl = params[p_idx]
        p_idx += 1
        lr_rl = 0.5 + 0.5 * erf(params[p_idx] / sqrt(2))
        p_idx += 1
    else
        beta_rl = 0.
        lr_rl = 0.
    end

    # Win-stay, lose-shift: beta (learning rate is 1)
    if include_wsls
        beta_wsls = params[p_idx]
        p_idx += 1
    else
        beta_wsls = 0.
    end

    # Initialize parameters
    lr_wsls = 1.
    Q_rl = zeros(typeof(beta_1),2)
    Q_wsls = zeros(typeof(beta_1),2)
    lik = 0.
    r::Array{Int64,1} = data.r
    c::Array{Int64,1} = data.c
    prevc = 0;
    unchosen_actions = [2,1];

    # Iterate over choices for the likelihood
    for i = 1:length(c)
        if (c[i]>0)

            Q = zeros(typeof(beta_1),2)
            Q[1] = beta_1 + beta_rl*Q_rl[1] + beta_wsls*Q_wsls[1]
            Q[2] = beta_rl*Q_rl[2] + beta_wsls*Q_wsls[2]

            if prevc>0
                Q[prevc] += ps;
            end

            lik += Q[c[i]] - logsumexp(Q)

            # RL update
            Q_rl[c[i]] = (1-lr_rl) * Q_rl[c[i]] + r[i]

            # WSLS update
            Q_wsls[c[i]] = (1-lr_wsls) * Q_wsls[c[i]] + r[i]
        end
    end

    return -lik
    
end

# Set up the different models
nll_neither = (params, data) -> qlik(params, data; include_rl = false, include_wsls = false)
nll_rl = (params, data) -> qlik(params, data; include_rl = true, include_wsls = false)
nll_wsls = (params, data) -> qlik(params, data; include_rl = false, include_wsls = true)
nll_rl_ws = (params, data) -> qlik(params, data; include_rl = true, include_wsls = true)

#89 (generic function with 1 method)

In [38]:
# Define all models for fitting
models_to_fit = [];

# Bias model
this_model = Dict()
this_model["model_name"] = "BIAS"
this_model["likfun"] = nll_neither
this_model["param_names"] = ["beta_1"]
push!(models_to_fit, this_model)

# Bias + RL model
this_model = Dict()
this_model["model_name"] = "BIAS_RL"
this_model["likfun"] = nll_rl
this_model["param_names"] = ["beta_1", "beta_rl", "lr_rl"]
push!(models_to_fit, this_model)

# Bias + WSLS model
this_model = Dict()
this_model["model_name"] = "BIAS_WSLS"
this_model["likfun"] = nll_wsls
this_model["param_names"] = ["beta_1", "beta_wsls"]
push!(models_to_fit, this_model)

# Bias + RL + WSLS model
this_model = Dict()
this_model["model_name"] = "BIAS_RL_WSLS"
this_model["likfun"] = nll_rl_ws
this_model["param_names"] = ["beta_1", "beta_rl", "lr_rl", "beta_wsls"]
push!(models_to_fit, this_model)

4-element Vector{Any}:
 Dict{Any, Any}("likfun" => var"#83#84"(), "param_names" => ["beta_1"], "model_name" => "BIAS")
 Dict{Any, Any}("likfun" => var"#85#86"(), "param_names" => ["beta_1", "beta_rl", "lr_rl"], "model_name" => "BIAS_RL")
 Dict{Any, Any}("likfun" => var"#87#88"(), "param_names" => ["beta_1", "beta_wsls"], "model_name" => "BIAS_WSLS")
 Dict{Any, Any}("likfun" => var"#89#90"(), "param_names" => ["beta_1", "beta_rl", "lr_rl", "beta_wsls"], "model_name" => "BIAS_RL_WSLS")

In [39]:
# Fit the models
for this_model in models_to_fit
    println("Fitting model: ", this_model["model_name"])
    fit_model_em(this_model, data_subset, to_save_folder; full = full, emtol = emtol, maxiter = 300, save_res = true, run_loo=false, comp_errors=true)
end


iter: 301
betas: [-1.96 0.02 -2.45 0.02]
sigma: [3.58, 0.04, 0.44, 0.06]
free energy: -29963.846764
change: [-4.0e-6, 0.000202, -0.000389, 0.000156, 1.7e-5, 0.001001, 0.001497, 0.001026]
max: 0.001497


In [40]:
# Load the results
model_results = []
for this_model in models_to_fit
    model_name = this_model["model_name"]
    res = load(joinpath(to_save_folder, model_name * ".jld2"))
    push!(model_results, res["model_res"])
end

In [65]:
# Make a table with the AIC, BIC, and NLL of each model (QG, 12481 players)
model_names = [this_model["model_name"] for this_model in models_to_fit]
model_iaic = [res["iaic"] for res in model_results]
model_ibic = [res["ibic"] for res in model_results]
model_ll  = [lml(res["x"],res["l"],res["h"]) for res in model_results]
model_table = DataFrame(model_name=model_names, iaic=model_iaic, ibic=model_ibic, nll=model_ll)

Row,model_name,iaic,ibic,nll
,String,Float64,Float64,Float64
1,BIAS,28100.8,28110.2,28098.8
2,BIAS_RL,27983.3,28011.5,27977.3
3,BIAS_WSLS,28046.2,28065.0,28042.2
4,BIAS_RL_WSLS,27926.2,27963.8,27918.2


In [41]:
# Make a table with the AIC, BIC, and NLL of each model (SD, 12863 players)
model_names = [this_model["model_name"] for this_model in models_to_fit]
model_iaic = [res["iaic"] for res in model_results]
model_ibic = [res["ibic"] for res in model_results]
model_ll  = [lml(res["x"],res["l"],res["h"]) for res in model_results]
model_table = DataFrame(model_name=model_names, iaic=model_iaic, ibic=model_ibic, nll=model_ll)

Row,model_name,iaic,ibic,nll
,String,Float64,Float64,Float64
1,BIAS,25140.9,25150.6,25138.9
2,BIAS_RL,24980.0,25009.0,24974.0
3,BIAS_WSLS,25076.5,25095.8,25072.5
4,BIAS_RL_WSLS,24948.5,24987.2,24940.5


In [20]:
# Make a table with the AIC, BIC, and NLL of each model (KK, 13886 players)
model_names = [this_model["model_name"] for this_model in models_to_fit]
model_iaic = [res["iaic"] for res in model_results]
model_ibic = [res["ibic"] for res in model_results]
model_ll  = [lml(res["x"],res["l"],res["h"]) for res in model_results]
model_table = DataFrame(model_name=model_names, iaic=model_iaic, ibic=model_ibic, nll=model_ll)

Row,model_name,iaic,ibic,nll
,String,Float64,Float64,Float64
1,BIAS,46134.8,46145.1,46132.8
2,BIAS_RL,44966.0,44996.8,44960.0
3,BIAS_WSLS,45773.2,45793.7,45769.2
4,BIAS_RL_WSLS,44508.1,44549.1,44500.1


In [34]:
# Make a table with the AIC, BIC, and NLL of each model (CK, 11963 players)
model_names = [this_model["model_name"] for this_model in models_to_fit]
model_iaic = [res["iaic"] for res in model_results]
model_ibic = [res["ibic"] for res in model_results]
model_ll  = [lml(res["x"],res["l"],res["h"]) for res in model_results]
model_table = DataFrame(model_name=model_names, iaic=model_iaic, ibic=model_ibic, nll=model_ll)

Row,model_name,iaic,ibic,nll
,String,Float64,Float64,Float64
1,BIAS,9906.34,9915.08,9904.34
2,BIAS_RL,9948.36,9974.58,9942.36
3,BIAS_WSLS,9914.88,9932.36,9910.88
4,BIAS_RL_WSLS,9948.44,9983.4,9940.44
